# COMP6010 Practical 03 — Bayesian Networks

**Work time:** 40 minutes  
**Discussion:** 20 minutes  
**Mini-project:** Represent a smart-home fault system as local CPTs and recover its joint distribution  
**Software:** NumPy, pandas, matplotlib

## Learning objectives
- encode a small DAG through local conditional probabilities
- calculate complete-world probabilities using BN factorization
- demonstrate conditional independence and explaining away

## Suggested timing
- **0–5 min:** answer the five review questions individually.
- **5–10 min:** run the setup and inspect the scenario/data.
- **10–35 min:** complete the implementation and experiment.
- **35–40 min:** record one result and one limitation.
- **40–60 min:** compare approaches and discuss the supplied prompts.

## Five key review questions

**Q1. What two components define a Bayesian network?**

> Your answer:  

1. A Directed Acyclic Graph (DAG) $G = (V, E)$, where nodes represent random variables and directed edges represent direct conditional dependencies.
2. A set of Conditional Probability Tables (CPTs) (or local distributions) $P(X_i \mid \text{Parents}(X_i))$ for every node $X_i$ given its parents in the DAG.

**Q2. How does a Bayesian network factorize a full joint distribution?**

> Your answer:  

By taking the product of the local conditional probability distributions of each variable conditioned on its parents: [ P(X_1, X_2, \dots, X_n) = \prod_{i=1}^{n} P(X_i \mid \text{Parents}(X_i)) ]

**Q3. What does an arrow normally assert, and what does it not automatically prove?**

> Your answer:  

* Asserts: A direct conditional dependence structure between parent and child nodes.
* Does NOT prove: Causal mechanisms in an experimental intervention sense, nor does it guarantee numerical dependence if CPT entries turn out to be uniform across parent configurations.

**Q4. What nodes belong to the Markov blanket of a variable?**

> Your answer:  

1. Its parents
2. Its children
3. Its children's other parents (co-parents)

Conditioning on all nodes in its Markov blanket isolates the variable, making it conditionally independent of all remaining nodes in the network.

**Q5. How does conditioning differ for a chain/fork and for a collider?**

> Your answer:

* Chain ($A \to B \to C$) & Fork ($A \leftarrow B \to C$): Unconditioned, $A$ and $C$ are dependent. Conditioning on $B$ blocks the path ($A \perp !!! \perp C \mid B$).
* Collider ($A \to B \leftarrow C$): Unconditioned, $A$ and $C$ are independent ($A \perp !!! \perp C$). Conditioning on $B$ (or any descendant of $B$) opens the path, introducing conditional dependence (explaining away).

## Scenario and graph

A smart-home monitoring system is represented by:

`Storm → PowerOutage → Alarm ← DeviceFault`  
`Alarm → PhoneAlert`

`Alarm` is a collider with two possible causes. We will enumerate the complete joint distribution from five small CPTs.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 6010
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=4, suppress=True)

In [2]:
from itertools import product

VARIABLES = ["Storm", "PowerOutage", "DeviceFault", "Alarm", "PhoneAlert"]

def p_true(variable, assignment):
    if variable == "Storm":
        return 0.15
    if variable == "PowerOutage":
        return 0.65 if assignment["Storm"] else 0.03
    if variable == "DeviceFault":
        return 0.08
    if variable == "Alarm":
        p, d = assignment["PowerOutage"], assignment["DeviceFault"]
        return {(True, True): 0.98, (True, False): 0.90,
                (False, True): 0.85, (False, False): 0.02}[(p, d)]
    if variable == "PhoneAlert":
        return 0.95 if assignment["Alarm"] else 0.04
    raise KeyError(variable)

def prob_value(variable, value, assignment):
    p = p_true(variable, assignment)
    return p if value else 1-p

### Task 1 — Complete the BN joint-probability function
For a full assignment, multiply the probability of each node value conditioned on its parents.

In [3]:
def joint_probability(assignment):
    prob = 1.0
    for var in VARIABLES:
        prob *= prob_value(var, assignment[var], assignment)
    return prob

### Task 2 — Enumerate all atomic events
There are 2⁵ = 32 complete assignments. Build a DataFrame and verify that the joint probabilities sum to 1.

In [4]:
# TODO: enumerate every Boolean assignment, calculate its probability, and create joint_df.
rows = []
for values in product([True, False], repeat=len(VARIABLES)):
    assignment = dict(zip(VARIABLES, values))
    p = joint_probability(assignment)
    row = dict(assignment)
    row["Probability"] = p
    rows.append(row)

joint_df = pd.DataFrame(rows)
joint_df

,Storm,PowerOutage,DeviceFault,Alarm,PhoneAlert,Probability
0,True,True,True,True,True,0.007262
1,True,True,True,True,False,0.000382
2,True,True,True,False,True,0.000006
3,True,True,True,False,False,0.000150
4,True,True,False,True,True,0.076694
5,True,True,False,True,False,0.004037
6,True,True,False,False,True,0.000359
7,True,True,False,False,False,0.008611
8,True,False,True,True,True,0.003391
9,True,False,True,True,False,0.000179


### Task 3 — Query the joint distribution
Implement a helper for P(Query=True | evidence) and compute P(PhoneAlert=True).

In [5]:
def conditional_probability(query, query_value=True, evidence=None):
    df = joint_df.copy()
    if evidence:
        for k, v in evidence.items():
            df = df[df[k] == v]
    total_prob = df["Probability"].sum()
    match_prob = df[df[query] == query_value]["Probability"].sum()
    return match_prob / total_prob

p_alert = conditional_probability("PhoneAlert", True)
p_alert

np.float64(0.21040660000000003)

### Task 4 — Independence before conditioning
Compare P(DeviceFault) with P(DeviceFault | Storm). The root variables were modelled independently.

In [6]:
# TODO: compute and compare the two probabilities.
p_d = conditional_probability("DeviceFault", True)
p_d_given_s = conditional_probability("DeviceFault", True, evidence={"Storm": True})

In [7]:
p_d

np.float64(0.08000000000000003)

In [8]:
p_d_given_s

np.float64(0.07999999999999999)

### Task 5 — Explaining away
Compare P(DeviceFault | Alarm) with P(DeviceFault | Alarm, Storm). Observing a likely alternative cause of the alarm should reduce the posterior for `DeviceFault`.

In [9]:
# TODO: demonstrate explaining away numerically.
p_d_given_a = conditional_probability("DeviceFault", True, evidence={"Alarm": True})
p_d_given_a_s = conditional_probability("DeviceFault", True, evidence={"Alarm": True, "Storm": True})

In [10]:
p_d_given_a

np.float64(0.3699626188187546)

In [11]:
p_d_given_a_s

np.float64(0.12069744914433322)

## Discussion — 20 minutes
1. Which assumptions in this graph are structural, and which are numerical?

* Structural: The directed graph topology (e.g., direct parent-child relationships like Storm $\to$ PowerOutage, independence between Storm and DeviceFault, and conditional independence of PhoneAlert given Alarm).
* Numerical: The specific baseline probability values assigned in the CPTs (e.g., $P(\text{Storm}) = 0.15$, $P(\text{DeviceFault}) = 0.08$, or $P(\text{Alarm} \mid \text{PowerOutage}, \text{DeviceFault}) = 0.98$).

2. Why does observing `Alarm` connect its otherwise independent causes?

* Alarm acts as a collider ($PowerOutage \to Alarm \leftarrow DeviceFault$). When the collider node is observed (Alarm = True), knowing whether one cause is present alters the likelihood of the other cause needed to account for the observed alarm.

3. How would the model change if a phone network outage could suppress `PhoneAlert`?

- A new parent node `PhoneNetworkOutage` would be added directly pointing to `PhoneAlert`:
  `Alarm → PhoneAlert ← PhoneNetworkOutage`

- `PhoneAlert` would become a collider node whose CPT depends on both `Alarm` and `PhoneNetworkOutage`.

## Evidence to retain for a practical portfolio
- the graph/CPT description
- a verified 32-row joint distribution
- the explaining-away calculation and interpretation